[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pythonanywhere/pypath/blob/main/notebooks/module6/01-opencv-basics.ipynb)

# Module 6.1 — OpenCV Basics
**Module 6: Computer Vision** | Estimated time: 20 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Read and display images using OpenCV and Matplotlib
- Understand color spaces (BGR, RGB, HSV) and convert between them
- Inspect image properties: shape, dtype, and size
- Apply basic geometric transformations: resize, flip, rotate, and transpose
- Draw shapes and text on images with OpenCV drawing functions
- Save processed images to disk

In [ ]:
# Install OpenCV headless (no GUI dependencies needed in Colab)
!pip install opencv-python-headless --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version: {np.__version__}')

# Helper function we will reuse throughout
def show(img, title='Image', bgr=True, figsize=(6, 4)):
    """Display a single image. Converts BGR→RGB automatically if bgr=True."""
    plt.figure(figsize=figsize)
    if bgr and len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    cmap = 'gray' if len(img.shape) == 2 else None
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Download a sample image (public domain landscape)
import requests
os.makedirs('/tmp/cv_images', exist_ok=True)
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
response = requests.get(url)
with open('/tmp/cv_images/sample.png', 'wb') as f:
    f.write(response.content)
print('Sample image saved to /tmp/cv_images/sample.png')

## Reading and Displaying Images

OpenCV uses `cv2.imread()` to load images from disk. By default it reads in **BGR** order (Blue-Green-Red), not the RGB that most other libraries expect. This is a common source of color problems — always remember to convert when displaying with Matplotlib.

In [ ]:
# Read the image
img_bgr = cv2.imread('/tmp/cv_images/sample.png')

if img_bgr is None:
    # Fallback: create a synthetic image so the notebook still works
    img_bgr = np.zeros((300, 400, 3), dtype=np.uint8)
    img_bgr[:, :200] = (255, 0, 0)   # Blue region (BGR)
    img_bgr[:, 200:] = (0, 0, 200)   # Red region (BGR)
    print('Using synthetic fallback image.')
else:
    print(f'Image loaded successfully')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_bgr)              # Raw BGR — colours will look wrong
axes[0].set_title('Raw BGR (wrong colours in matplotlib)')
axes[0].axis('off')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
axes[1].imshow(img_rgb)              # Correct
axes[1].set_title('Converted to RGB (correct)')
axes[1].axis('off')

plt.suptitle('BGR vs RGB in Matplotlib', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Image Properties

Every loaded image is a **NumPy array**. The key attributes are:

| Attribute | Meaning |
|---|---|
| `shape` | `(height, width, channels)` for colour, `(height, width)` for grayscale |
| `dtype` | Usually `uint8` (values 0–255) |
| `size` | Total number of elements (H × W × C) |

In [ ]:
print('=== Image Properties ===')
print(f'Shape (H, W, C) : {img_bgr.shape}')
print(f'Height          : {img_bgr.shape[0]} px')
print(f'Width           : {img_bgr.shape[1]} px')
print(f'Channels        : {img_bgr.shape[2]}')
print(f'Data type       : {img_bgr.dtype}')
print(f'Total elements  : {img_bgr.size:,}')
print(f'Memory (bytes)  : {img_bgr.nbytes:,}')
print(f'Min pixel value : {img_bgr.min()}')
print(f'Max pixel value : {img_bgr.max()}')
print(f'Mean pixel value: {img_bgr.mean():.2f}')

# Accessing individual pixels (row, col) — returns [B, G, R]
px = img_bgr[10, 10]
print(f'\nPixel at (10,10) BGR: {px}')

## Color Spaces: BGR, RGB, and HSV

OpenCV supports many color spaces. The most important for practical work are:

- **BGR / RGB** — the familiar red-green-blue model
- **HSV** — Hue, Saturation, Value. Hue encodes the colour type (0–179 in OpenCV), Saturation encodes the vibrancy (0–255), and Value encodes the brightness (0–255). HSV is very useful for colour-based object detection because it separates colour information from lighting.

In [ ]:
# Convert to several color spaces
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2Lab)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0, 0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original (RGB)')

axes[0, 1].imshow(img_gray, cmap='gray')
axes[0, 1].set_title('Grayscale')

axes[0, 2].imshow(img_hsv)   # channels displayed as-is for visualisation
axes[0, 2].set_title('HSV (raw array)')

# Split HSV into individual channels
h, s, v = cv2.split(img_hsv)
axes[1, 0].imshow(h, cmap='hsv')
axes[1, 0].set_title('H — Hue (colour type)')

axes[1, 1].imshow(s, cmap='gray')
axes[1, 1].set_title('S — Saturation (vibrancy)')

axes[1, 2].imshow(v, cmap='gray')
axes[1, 2].set_title('V — Value (brightness)')

for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Color Space Exploration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Practical HSV example: isolate a colour range
print('HSV range for "red" in OpenCV:')
print('  Lower: H=0..10,   S=100..255, V=100..255')
print('  Upper: H=160..179, S=100..255, V=100..255')

## Basic Geometric Transformations

OpenCV provides fast C-backed implementations of common spatial transforms:

| Function | Purpose |
|---|---|
| `cv2.resize()` | Scale image to a new size |
| `cv2.flip()` | Mirror horizontally (1), vertically (0), or both (-1) |
| `cv2.rotate()` | Rotate in 90° increments |
| `cv2.warpAffine()` | Arbitrary affine transform (uses rotation matrix) |

In [ ]:
h, w = img_bgr.shape[:2]

# Resize
img_half  = cv2.resize(img_bgr, (w // 2, h // 2))       # half size
img_fixed = cv2.resize(img_bgr, (300, 200))               # fixed size
img_scale = cv2.resize(img_bgr, None, fx=1.5, fy=1.5,
                       interpolation=cv2.INTER_LINEAR)     # scale factor

# Flip
img_hflip = cv2.flip(img_bgr, 1)   # horizontal (mirror)
img_vflip = cv2.flip(img_bgr, 0)   # vertical (upside-down)

# Rotate using rotation matrix (arbitrary angle)
center = (w // 2, h // 2)
M = cv2.getRotationMatrix2D(center, angle=30, scale=1.0)
img_rot30 = cv2.warpAffine(img_bgr, M, (w, h))

# Rotate in 90-degree steps
img_rot90 = cv2.rotate(img_bgr, cv2.ROTATE_90_CLOCKWISE)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
data = [
    (img_bgr,    'Original'),
    (img_half,   f'Resized ½ ({w//2}×{h//2})'),
    (img_hflip,  'Flip Horizontal'),
    (img_vflip,  'Flip Vertical'),
    (img_rot30,  'Rotate 30°'),
    (img_rot90,  'Rotate 90° CW'),
]
for ax, (im, title) in zip(axes.flat, data):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')

plt.suptitle('Geometric Transformations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Drawing on Images

OpenCV has a family of drawing functions. They all modify the image **in place** and use BGR colour tuples.

```python
cv2.rectangle(img, pt1, pt2, color, thickness)
cv2.circle(img, center, radius, color, thickness)  # thickness=-1 fills
cv2.line(img, pt1, pt2, color, thickness)
cv2.putText(img, text, org, fontFace, fontScale, color, thickness)
```

In [ ]:
# Work on a copy so we don't modify the original
canvas = img_bgr.copy()

# Rectangle (bounding-box style)
cv2.rectangle(canvas, (20, 20), (120, 90), (0, 255, 0), thickness=3)

# Filled circle
cv2.circle(canvas, (200, 60), radius=40, color=(0, 0, 255), thickness=-1)

# Hollow circle
cv2.circle(canvas, (200, 60), radius=40, color=(255, 255, 255), thickness=2)

# Diagonal line
cv2.line(canvas, (0, 0), (w - 1, h - 1), color=(255, 0, 0), thickness=2)

# Text annotation
cv2.putText(
    canvas,
    text='OpenCV Drawing!',
    org=(10, h - 15),
    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
    fontScale=0.7,
    color=(255, 255, 0),
    thickness=2,
    lineType=cv2.LINE_AA  # anti-aliased
)

# Arrow
cv2.arrowedLine(canvas, (w // 2, 20), (w // 2, 80), (0, 165, 255), thickness=3)

show(canvas, 'Drawing Shapes and Text')

## Saving Images

`cv2.imwrite()` saves an image to disk. It automatically infers the format from the file extension (`.jpg`, `.png`, `.bmp`, etc.). For JPEG, you can control quality; for PNG, compression level.

In [ ]:
# Save the annotated image as PNG (lossless)
out_png = '/tmp/cv_images/annotated.png'
success = cv2.imwrite(out_png, canvas)
print(f'PNG saved: {success} → {out_png}')
print(f'File size: {os.path.getsize(out_png):,} bytes')

# Save as JPEG with quality=85
out_jpg = '/tmp/cv_images/annotated_q85.jpg'
cv2.imwrite(out_jpg, canvas, [cv2.IMWRITE_JPEG_QUALITY, 85])
print(f'JPEG (q=85) size: {os.path.getsize(out_jpg):,} bytes')

# Save grayscale
out_gray = '/tmp/cv_images/grayscale.png'
cv2.imwrite(out_gray, img_gray)
print(f'Grayscale PNG saved → {out_gray}')

# Verify by reloading
verify = cv2.imread(out_png)
print(f'\nReloaded shape: {verify.shape}  (should match original {img_bgr.shape})')

## Summary

| Concept | Key function / note |
|---|---|
| Read image | `cv2.imread(path)` returns BGR NumPy array |
| Display | Convert BGR→RGB before `plt.imshow()` |
| Color convert | `cv2.cvtColor(img, cv2.COLOR_BGR2HSV)` |
| Resize | `cv2.resize(img, (w, h))` or `fx/fy` scale |
| Flip | `cv2.flip(img, 1)` horizontal, `0` vertical |
| Rotate | `cv2.rotate()` or `cv2.warpAffine()` with rotation matrix |
| Draw | `cv2.rectangle / circle / line / putText` |
| Save | `cv2.imwrite(path, img)` |

## Practice Exercises

**Exercise 1 — Color Masking:**  
Load any image, convert it to HSV, and use `cv2.inRange()` to create a binary mask that isolates a specific colour (e.g., green plants or a blue sky). Apply the mask with `cv2.bitwise_and()` and display the result.

**Exercise 2 — Transformation Pipeline:**  
Take the sample image, resize it to 256×256, rotate it 45°, flip it horizontally, and overlay the original and transformed versions side by side using `np.hstack()`. Add a text label to each half.

**Exercise 3 — Annotated Thumbnail Generator:**  
Write a function `annotate_thumbnail(path, label)` that reads an image, resizes it to 200×200, draws a coloured border rectangle, overlays the label text in the bottom-left corner, and saves the result to `/tmp/` returning the output path.